# Class 1 — Why Retrieval-Augmented Generation?

**Week 6: Foundations of RAG and Chatbots**

### Learning objectives
By the end of this notebook you will be able to:
- Explain why every LLM has a hard "knowledge cutoff" and why that leads to hallucination
- Distinguish parametric knowledge (baked into model weights) from retrieved knowledge (supplied at request time)
- Manually "ground" a prompt by pasting real context into it, and see the difference it makes
- Describe, at a conceptual level, the end-to-end RAG pipeline you'll build for real in Class 3

> This class is intentionally light on new libraries. The point is to *feel* the problem RAG solves before we build any retrieval machinery next class.

## Setup

Run the cell below first. It reads your API key from the environment — never hardcode a key in a notebook.
You only need **one** of `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` set.

In [ ]:
import os

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")

if not OPENAI_API_KEY and not ANTHROPIC_API_KEY:
    print(
        "No API key found in your environment.\n"
        "Set one of the following before running the demo cells below:\n"
        "  export OPENAI_API_KEY=sk-...\n"
        "  export ANTHROPIC_API_KEY=sk-ant-...\n"
        "The rest of this notebook will still run and print helpful guidance, "
        "but the actual model calls will raise an error until a key is set."
    )
else:
    provider = "Anthropic" if ANTHROPIC_API_KEY else "OpenAI"
    print(f"Found a key for {provider}. You're ready to run the demo cells below.")

### A tiny helper

We'll reuse this one function all week: it sends a prompt to whichever provider you have a key for.
If both keys are set, it prefers Anthropic — feel free to flip that if you'd rather default to OpenAI.

In [ ]:
def call_llm(prompt, system=None, model_openai="gpt-4o-mini", model_anthropic="claude-3-5-haiku-20241022"):
    """Send a single prompt to whichever provider has a key configured.

    Returns the model's text reply as a plain string.
    Raises RuntimeError if neither OPENAI_API_KEY nor ANTHROPIC_API_KEY is set.
    """
    if ANTHROPIC_API_KEY:
        import anthropic
        client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
        response = client.messages.create(
            model=model_anthropic,
            max_tokens=400,
            system=system or "You are a helpful, concise assistant.",
            messages=[{"role": "user", "content": prompt}],
        )
        return response.content[0].text

    elif OPENAI_API_KEY:
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY)
        messages = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": prompt})
        response = client.chat.completions.create(
            model=model_openai,
            messages=messages,
        )
        return response.choices[0].message.content

    else:
        raise RuntimeError(
            "No API key configured. Set OPENAI_API_KEY or ANTHROPIC_API_KEY and re-run the Setup cell."
        )

## Concept 1 — Ask the Model Something It Can't Know

`Nimbus Robotics` is a company we just invented for this notebook. It exists nowhere on the internet, so no
LLM has ever been trained on its employee handbook. Let's ask about it anyway and see what happens.

In [ ]:
question = (
    "According to Nimbus Robotics' internal employee handbook, "
    "how many paid volunteer days does each employee get per year?"
)

answer = call_llm(question)
print(answer)

Notice that the model almost never says "I don't know" outright — it tends to produce a fluent, specific-sounding
number anyway. That's **hallucination**: a confident answer with nothing real behind it. The model isn't lying on
purpose; it genuinely has no way to distinguish "a fact I remember" from "a plausible-sounding pattern I can generate."

## Concept 2 — Now Hand It the File

This time we paste the real (fictional, but fixed) policy directly into the prompt as **context**, and instruct the
model to answer only from that context. This is manual grounding — the simplest possible form of RAG, with no
retrieval step at all because we're handing over the one paragraph that matters ourselves.

In [ ]:
context = """Nimbus Robotics Employee Handbook, Section 4.2 -- Volunteer Time Off:
Every full-time employee receives 3 paid volunteer days per calendar year, in addition to
standard PTO. Volunteer days do not roll over to the next year and must be scheduled with
a manager at least one week in advance.
"""

grounded_prompt = f"""Use ONLY the context below to answer the question.
If the answer is not contained in the context, say you don't know.

CONTEXT:
{context}

QUESTION: {question}
"""

grounded_answer = call_llm(grounded_prompt)
print(grounded_answer)

Same model, same question — the only thing that changed between the two calls above is what the model was
allowed to read. That single change is the entire idea behind RAG. Everything else we build over the next two
classes (chunking, embeddings, vector search) exists purely to **automate finding the right paragraph to paste**,
because you can't do this by hand once you have thousands of documents.

## Concept 3 — The RAG Architecture, End to End

At a high level, a RAG system has two phases:

```
INDEXING (done once, or whenever documents change)
  Documents -> Chunk -> Embed -> Vector Store

QUERY TIME (done on every user question)
  Question -> Embed -> Retrieve top-k chunks -> Augmented Prompt -> LLM -> Answer
```

We'll build the "Embed" and "Retrieve top-k" steps for real in Class 2, and assemble the complete pipeline in
Class 3. For now, the important takeaway is: **RAG is grounding, automated at scale.**

## Challenges

Work through these in order. Each one builds on the `call_llm` helper and the grounding pattern above.
No solutions are provided — write your own code in the empty cells below each prompt.

### Challenge 1 — Catch It Guessing

Pick a fact the model cannot possibly know: something about a fictional entity you invent, or a very recent event.
Ask an **ungrounded** question about it with `call_llm` and print the answer.

**Acceptance criteria:** your printed answer contains a specific, fabricated-sounding detail (a number, a name, a
date) that you can prove is made up, because you invented the entity yourself.

In [ ]:
# TODO: invent a fact the model can't know, ask it, and print the (likely hallucinated) answer


### Challenge 2 — Write Your Own Grounding Paragraph

Write a short (3-6 sentence) paragraph of "real" context about your invented fact from Challenge 1 — the actual
correct answer. Build a grounded prompt like the one in Concept 2 and print the model's grounded answer.

**Acceptance criteria:** the grounded answer correctly reflects the paragraph you wrote, not the guess from
Challenge 1.

In [ ]:
# TODO: write a context paragraph, build a grounded prompt, and print the grounded answer


### Challenge 3 — Compare Side by Side

Print both answers (ungrounded from Challenge 1, grounded from Challenge 2) together with clear labels, so the
contrast is obvious at a glance.

**Acceptance criteria:** your output clearly shows two labeled answers to the same question, one wrong and one
correct.

In [ ]:
# TODO: print the ungrounded and grounded answers together, clearly labeled


### Challenge 4 — Diagram the Pipeline in Your Own Words

In the code cell below, write a Python comment (or a triple-quoted string) that redraws the RAG pipeline diagram
from Concept 3 using your own labels and a real example from your own life or work (e.g. a folder of your own
notes, a customer support inbox, a wiki).

**Acceptance criteria:** your diagram names a concrete "documents" source and a concrete "question" someone might
ask about it.

In [ ]:
# TODO: write your own version of the indexing/query-time diagram as a comment or string


### Challenge 5 — Bonus: Spot the Failure Mode

Below is a transcript of a broken RAG-style exchange. In a comment, name which failure mode from this class it
most resembles (knowledge cutoff, no retrieval at all, wrong chunk retrieved, or over-long irrelevant context) and
explain your reasoning in 2-3 sentences.

```
CONTEXT: [50 paragraphs about the company's marketing history, none about vacation policy]
QUESTION: How many vacation days do new hires get?
ANSWER: Based on the context, new hires typically receive around 15 vacation days, though this may vary.
```

**Acceptance criteria:** your comment names one specific failure mode and justifies the choice using details from
the transcript.

In [ ]:
# TODO: name the failure mode shown above and justify your answer in a comment
